[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhaskarjitsarmah/RL-Agents-Workshop-LLM/blob/main/notebooks/NB5_openpipe_art.ipynb)

# NB5 - The Same Experiment, in OpenPipe ART

Count what we hand-rolled in NB3 and NB4: rollout collection and batching,
reward-to-trajectory plumbing, advantage computation, per-turn loss masking,
checkpointing, and a serving path we still do not have.

That is infrastructure, not research - and it is the part most likely to hide a
quiet bug that costs a week.

**ART** (OpenPipe's Agent Reinforcement Trainer) owns all of it: GRPO + LoRA +
vLLM + W&B behind a client/server split, multi-turn-native rather than bolted on.

We run the *same* experiment - same tasks, same reward, same prompts - and
compare both the curves and the line count. The framework is the only variable.

> **Restart the runtime before this notebook.** Colab does not free GPU memory between notebooks, and a leftover model from the previous one is the most common cause of an out-of-memory error halfway through a training run.
>
> *Runtime -> Restart session*, then run the setup cell below.

In [ ]:
# --- Setup. Safe to re-run. ---------------------------------------------
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/RL-Agents-Workshop-LLM"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/bhaskarjitsarmah/RL-Agents-Workshop-LLM.git", "/content/RL-Agents-Workshop-LLM"], check=True)
    os.chdir("/content/RL-Agents-Workshop-LLM")
    # Colab's own keyring, read BEFORE preflight computes CAP -- otherwise the
    # notebook decides "no W&B key" while the key sits unread in the sidebar.
    # Absent secrets are normal, not an error: everything downgrades gracefully.
    try:
        from google.colab import userdata
        for _k in ("WANDB_API_KEY", "OPENAI_API_KEY", "HF_TOKEN"):
            try:
                os.environ.setdefault(_k, userdata.get(_k) or "")
            except Exception:
                pass
    except Exception:
        pass
    for _k in [k for k, v in list(os.environ.items()) if v == ""]:
        del os.environ[_k]          # empty != set; CAP tests truthiness

    # Install with uv, not pip: same resolution, several times faster on Colab.
    # The CORE layers on top of Colab's torch and never replaces it -- see the
    # header of requirements-colab.txt for why pinning torch broke this before.
    print("Installing the training stack with uv (1-2 min the first time)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

    def _uv(*pkgs):
        return subprocess.run([sys.executable, "-m", "uv", "pip", "install",
                               "--system", "-q", *pkgs]).returncode

    # CORE -- must succeed. Fail loudly instead of continuing on stock packages:
    # a swallowed install failure surfaces 10 cells later as a dtype or
    # bitsandbytes error that names nothing resembling its cause.
    if _uv("-r", "requirements-colab.txt") != 0:
        print("*** CORE INSTALL FAILED -- scroll up for the uv error. ***")
        raise SystemExit("core install failed -- see WORKSHOP_GUIDE.md")

    # NB5's openpipe-art, best-effort: it can fail to resolve, and NB5 falls back
    # to a pre-baked run if it is absent. A failure here must not break the core.
    _uv("openpipe-art>=0.4.0")

    # Unsloth: ~2x faster LoRA on a T4, which is the difference between NB3
    # fitting in a lunch break and not. Installed HERE rather than in
    # requirements-colab.txt, and UNPINNED.
    #   * unpinned, because the old `unsloth==2024.12.4` pin required torch
    #     2.5.1 and was what made the entire install abort;
    #   * here rather than in the requirements file, because this is the one
    #     dependency heavy enough to fail on the day, and a failure has to
    #     degrade to the transformers + bitsandbytes path, not kill the core.
    # Skip it with:  os.environ["USE_UNSLOTH"] = "0"  above this cell.
    if os.environ.get("USE_UNSLOTH") != "0":
        if _uv("unsloth", "unsloth_zoo") != 0:
            print("unsloth did not install -- continuing on transformers + "
                  "bitsandbytes. Same adapter, slower. This is not an error.")

    # Unsloth CAN drag a different torch in. If it did, the kernel must restart
    # before anything imports torch, or you get a cryptic CUDA error later.
    from importlib.metadata import version as _ver
    if "torch" in sys.modules and sys.modules["torch"].__version__ != _ver("torch"):
        print("=" * 68)
        print("  torch was replaced. Runtime -> Restart session, then Run all again.")
        print("=" * 68)
        raise SystemExit("restart required -- see the message above")
else:
    # Run from the REPO ROOT in both environments, so every relative path in
    # every notebook ("data/...") means the same thing whether you are on Colab
    # (cwd = repo root) or opened the file locally from notebooks/.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
sys.path.insert(0, os.getcwd())

# Results that outlive the VM. Every Colab notebook is a SEPARATE runtime, so
# NB3 trains the GRPO curve into its own /content and NB5 -- a different VM --
# cannot see it. Anything one notebook computes for another has to land
# somewhere shared, and Drive is the only such place on free Colab.
# Set RESULTS_DIR before importing llm_utils: it is read at import time.
if IN_COLAB and os.environ.get("USE_DRIVE", "1") != "0":
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        _rd = "/content/drive/MyDrive/rl-workshop-results"
        os.makedirs(_rd, exist_ok=True)
        os.environ["RESULTS_DIR"] = _rd
        print(f"Results -> {_rd} (shared across notebooks, survives restarts)")
    except Exception as _e:
        print(f"Drive not mounted ({_e}). Results stay in this VM only, so a")
        print("later notebook will not see what this one computes. Not fatal.")

from llm_utils import (build_db, preflight, capability, load_result,
                       report_number, save_result)
from llm_utils.plotting import use_house_style
import matplotlib.pyplot as plt

CAP = preflight()
use_house_style()
print("Database ready at:", build_db())
if not CAP["gpu"]:
    print()
    print("No GPU detected -> REPLAY MODE.")
    print("Training cells will load pre-baked runs; every chart still renders.")
    print("In Colab: Runtime -> Change runtime type -> T4 GPU, then re-run.")

if not CAP["wandb"]:
    os.environ.setdefault("WANDB_MODE", "offline")
    print()
    print("No WANDB_API_KEY -> W&B set to offline mode.")
    print("Training still runs and still logs; the curves land in ./wandb")
    print("instead of the cloud dashboard.")

In [ ]:
def baked(key, how):
    """Load a pre-baked run, or explain exactly how to produce it.

    Returns None when the artifact is missing. Callers must check -- we would
    rather show no chart than an invented one.
    """
    data = load_result(key)
    if data is None:
        print(f"[{key}] not baked yet.")
        print(f"  Produce it with:  {how}")
        print("  Then re-run this cell. (The pre-baked files ship with the repo;")
        print("   you only need this if you are rebuilding them yourself.)")
    return data


PREBAKED = not CAP["gpu"]   # charts get a watermark when we are replaying

## 0. Probe the API before trusting anything

ART moves faster than TRL, and this repo was written against a version that may
not be the one you just installed.

**Run this first and treat its output as ground truth.** If a name is missing,
find its real import path before running the live cells - do not assume the
notebook is right and the library is wrong.

In [ ]:
from llm_utils.art_bridge import art_available, art_probe

info = art_probe()
ART_OK = info.get("available") and not info.get("missing_expected")
print("\nlive ART cells enabled:", bool(ART_OK))

> **If ART does not initialise here, nothing is lost.** Its local backend wants
> vLLM, and vLLM on a Turing-class T4 is fragile. The notebook falls back to a
> pre-baked run recorded on an A10, clearly watermarked, and still ships the
> exact code you would run on a bigger GPU.
>
> That fallback is a deliberate design decision, not an apology: a workshop
> should not stake a module on a library initialising on free-tier hardware.

## 1. The ART loop

Four objects, and each maps onto something we built by hand:

| ART | ours (NB3/NB4) |
|---|---|
| `art.TrainableModel` | the 4-bit policy + LoRA from `load_4bit_policy` |
| `art.Trajectory` | `rollout.Trajectory` |
| `art.TrajectoryGroup` | the list returned by `rollout_group` |
| `await model.train(groups)` | `GRPOTrainer.train()` |

Note what is *absent*: we never compute an advantage. ART does that, because the
group is a first-class object rather than a batching detail.

In [ ]:
import inspect
from llm_utils import art_bridge
print(inspect.getsource(art_bridge.art_rollout))

The reward and the prompt come from **our** modules, not ART's. That is
deliberate: NB5 compares TRL-GRPO against ART-GRPO, and the comparison only
means something if the reward and the prompt are byte-identical on both sides.

In [ ]:
from llm_utils.gen_tasks import read_jsonl

train = read_jsonl("data/tasks_train_gen.jsonl")
art_hist = None

if ART_OK and CAP["gpu"]:
    from llm_utils.art_bridge import run_art_training
    try:
        out = await run_art_training(train, steps=20, groups_per_step=8,
                                     rollouts_per_group=8)
        art_hist = out["history"]
        save_result("nb5_art_history", art_hist)
    except Exception as e:
        # ART imports cleanly and still cannot train here: its local backend
        # pulls in megatron/vLLM, which a Turing T4 does not have. Reaching this
        # line is the finding, not an accident -- so it must not take the rest
        # of the notebook down with it.
        print(f"{type(e).__name__}: {e}\n")
        art_hist = baked("nb5_art_history",
                  "python scripts/bake_all.py --stage art")
else:
    art_hist = baked("nb5_art_history",
                  "python scripts/bake_all.py --stage art")

## 2. TRL vs ART, same reward, same data

In [ ]:
trl_hist = baked("nb3_grpo_history",
                  "python scripts/bake_all.py --stage grpo")
# Draw whichever curves exist. Requiring BOTH meant that on a T4 -- where ART
# cannot train at all -- this cell showed nothing, including the TRL run that
# NB3 really did produce. One measured line with the gap named is a better
# comparison than an empty panel.
if trl_hist or art_hist:
    fig, ax = plt.subplots(figsize=(9, 4))
    if trl_hist:
        pts = [(h["step"], h["reward"]) for h in trl_hist if "reward" in h]
        ax.plot([p[0] for p in pts], [p[1] for p in pts],
                label="TRL GRPOTrainer (NB3)", color="#4C72B0", lw=1.8)
    if art_hist:
        ax.plot([h["step"] for h in art_hist], [h["reward"] for h in art_hist],
                label="OpenPipe ART (this notebook)", color="#DD8452", lw=1.8)
    else:
        ax.plot([], [], color="#DD8452", lw=1.8, ls="--",
                label="OpenPipe ART -- not run (needs megatron/vLLM)")
        ax.text(0.5, 0.5, "ART line missing:\nlocal backend needs a\nmegatron/vLLM stack "
                          "this T4\ncannot provide",
                transform=ax.transAxes, ha="center", va="center",
                fontsize=9, color="#8C8C8C", style="italic")
    ax.set_xlabel("step"); ax.set_ylabel("mean reward")
    ax.set_title("Same reward, same data, same prompts -- only the framework differs")
    ax.legend()
    if PREBAKED:
        ax.text(0.99, 0.02, "pre-baked replay", transform=ax.transAxes,
                ha="right", fontsize=8, color="#8C8C8C", style="italic")
    plt.tight_layout(); plt.show()
    if not art_hist:
        print("Only the TRL curve is measured here. The ART line is what you")
        print("would get on hardware that can host its local backend -- the")
        print("code above is unchanged and is what you would run there.")

## 3. The punchline

ART serves the trained policy behind an **OpenAI-compatible** endpoint. So the
adapter we just trained can be handed to the *vendored, unmodified* agent and
scored by the *vendored, unmodified* `evaluate()`:

```python
evaluate(art_openai_agent(model), split="test")
```

Three different backends now - OpenAI, a local 4-bit Qwen, and an ART-served
policy - measured by one function on one set of 16 tasks.

This is the entire reason `agents.py` got its single `llm_fn` hook back in Phase
0. Continuity with repo 1, made literal.

In [ ]:
from llm_utils import evaluate

if ART_OK and CAP["gpu"]:
    from llm_utils.art_bridge import art_openai_agent
    res_art = evaluate(art_openai_agent(out["model"]), split="test")
    print(report_number(res_art, "ART-trained policy"))
else:
    res_art = baked("nb5_art_eval",
                  "python scripts/bake_all.py --stage art")
    if res_art:
        print(report_number(tuple(res_art["test16"]), "ART-trained policy"))

## 4. What ART owns for you

In [ ]:
from llm_utils.art_bridge import art_notes
print(art_notes())

### And when you have no verifiable reward

Everything in this workshop rests on `score_sql`: execute both queries, compare
result sets. Most agent tasks have nothing like it.

ART's answer is **RULER** - a general-purpose LLM judge that scores trajectories
*relative to each other within a group*, which is exactly the comparison GRPO
needs. It is strictly worse than a verifiable reward (you inherit the judge's
biases, and NB6's failure mode gets much easier to hit), but it is the
difference between "we can run RL on this" and "we cannot".

We did not need it. Know that it exists for when you do.

## Takeaways

1. ART is GRPO + LoRA + vLLM + W&B with a client/server split, and it is **multi-turn-native** rather than patched.
2. Because it serves an OpenAI-compatible endpoint, the **vendored agent and the vendored `evaluate()` work against it unchanged** - a third backend on the same scoreboard.
3. Keep the reward and the prompt in *your* code, not the framework's. Otherwise a framework comparison quietly becomes a reward comparison.
4. **Probe the API before trusting the notebook.** `art_probe()` prints the real namespace; this file was written against a version that may not be yours.
5. No verifiable reward? RULER. Worse than execution matching, far better than nothing.

### The gap this leaves (-> NB6)

Every number so far has assumed that the reward we optimised is the reward we
wanted. On this task that assumption happens to hold, because execution matching
is genuinely what we care about.

Now suppose it did not. Suppose - as on most real tasks - you had to write a
plausible-looking proxy instead. What would the optimizer do with it?

### Exercise

1. Run the ART loop with `rollouts_per_group=4` and `=16`. Where does the
   wall-clock go, and does the reward curve justify it?
2. Wire `r_hackable_rowcount` in as the ART reward instead of
   `composite_reward` and run 10 steps. You have just built NB6's demo through a
   different framework - does the framework protect you? (It does not.)

In [ ]:
# --- Cost / throughput meter -------------------------------------------
from llm_utils import METER, flush
print(METER)          # OpenAI spend (0 unless you ran the comparison rows)
try:
    import wandb; wandb.finish()
except Exception:
    pass
flush()